# HeatLab - neural network vs LightGBM (GPU)
Predicts the street-level 2 m air-temperature **anomaly** (station minus ERA5) from ERA5 weather, time of day/year and
real surroundings data (OSM fractions + ESA WorldCover + Copernicus DSM).

* Same 5-fold **spatial-block** cross-validation as the LightGBM study (whole station clusters held out, 0.5 km buffer).
* Residual MLP with three outputs: mean, 10 % and 90 % quantile (80 % prediction interval).
* Inputs are standardised with statistics from the **training rows of each fold only**.
* Fixed hyper-parameters (no tuning on test folds).

Upload `era5_ctx.parquet`, `stations.parquet` and `obs.parquet` (or `obs.parquet.part0/1`) to `/content` before running.

In [ ]:
import subprocess, os, time, json, numpy as np, pandas as pd, torch, torch.nn as nn
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout.strip() or 'NO GPU')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '| gpus:', torch.cuda.device_count(), '| torch', torch.__version__)

In [ ]:
import glob
parts = sorted(glob.glob('/content/obs.parquet.part*'))
if parts:                                   # obs.parquet was uploaded in chunks (upload limit): reassemble
    with open('/content/obs.parquet', 'wb') as o:
        for p in parts: o.write(open(p, 'rb').read())
era = pd.read_parquet('/content/era5_ctx.parquet'); st = pd.read_parquet('/content/stations.parquet'); ob = pd.read_parquet('/content/obs.parquet')
b = ob.merge(st, on=['city', 'station']).merge(era, on=['city', 'time'])
b['anom'] = b['anom_c'].astype(np.float32) / 100
assert len(b) == len(ob) == 1336694, (len(b), len(ob))
meta = ['city','station','time','anom','anom_c','fold','tr0','tr1','tr2','tr3','tr4']
ALL = [c for c in b.columns if c not in meta]
CTX = [c for c in ALL if c.startswith('e_')] + ['hod_sin','hod_cos','doy_sin','doy_cos','is_night']
RAS = [c for c in ALL if c.startswith('wc_') or c.startswith('dsm_')]
STATIC_OSM = [c for c in ALL if c not in CTX and c not in RAS]
b['tr0'] = b['tr0']; b['fold'] = b['fold'].astype(int)
SETS = {'osm_js': CTX + STATIC_OSM, 'osm_js+raster': CTX + STATIC_OSM + RAS, 'no_static': CTX}
print({k: len(v) for k, v in SETS.items()}, b.shape)

In [ ]:
class Block(nn.Module):
    def __init__(s, w, p):
        super().__init__(); s.n = nn.LayerNorm(w); s.f = nn.Sequential(nn.Linear(w, w*2), nn.GELU(), nn.Dropout(p), nn.Linear(w*2, w))
    def forward(s, x): return x + s.f(s.n(x))
class Net(nn.Module):
    def __init__(s, d_in, w=256, depth=4, p=0.10):
        super().__init__(); s.inp = nn.Linear(d_in, w); s.blocks = nn.Sequential(*[Block(w, p) for _ in range(depth)])
        s.out = nn.Sequential(nn.LayerNorm(w), nn.Linear(w, 3))   # mean, q10, q90
    def forward(s, x): return s.out(s.blocks(s.inp(x)))
def pinball(y, q, tau): d = y - q; return torch.maximum(tau*d, (tau-1)*d).mean()
def loss_fn(o, y): return nn.functional.mse_loss(o[:,0], y) + 0.5*(pinball(y, o[:,1], 0.1) + pinball(y, o[:,2], 0.9))

In [ ]:
EPOCHS, BS, LR, WD, SEED = 25, 4096, 2e-3, 1e-4, 7
def run_fold(feats, f):
    torch.manual_seed(SEED); np.random.seed(SEED)
    tr = (b[f'tr{f}'].to_numpy() == 1) & (b['time'].dt.hour.to_numpy() % 2 == 0)   # same row thinning as LightGBM
    te = (b['fold'].to_numpy() == f)
    Xtr = b.loc[tr, feats].to_numpy(np.float32); mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr = np.nan_to_num((Xtr - mu) / sd); Xte = np.nan_to_num((b.loc[te, feats].to_numpy(np.float32) - mu) / sd)
    ytr = b.loc[tr, 'anom'].to_numpy(np.float32)
    Xg, yg = torch.from_numpy(Xtr).to(dev), torch.from_numpy(ytr).to(dev)
    net = Net(len(feats)).to(dev)
    if dev == 'cuda' and torch.cuda.device_count() > 1: net = nn.DataParallel(net)      # multi-GPU path (data parallel)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WD)
    steps = EPOCHS * ((len(Xg) + BS - 1) // BS)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps)
    scaler = torch.amp.GradScaler(enabled=(dev == 'cuda'))
    t0 = time.time()
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(len(Xg), device=dev); tot = 0.0
        for i in range(0, len(Xg), BS):
            idx = perm[i:i+BS]
            with torch.autocast(device_type=dev, dtype=torch.float16, enabled=(dev == 'cuda')):
                l = loss_fn(net(Xg[idx]).float(), yg[idx])
            opt.zero_grad(set_to_none=True); scaler.scale(l).backward(); scaler.step(opt); scaler.update(); sch.step(); tot += l.item() * len(idx)
        if ep % 5 == 4 or ep == EPOCHS - 1: print(f'  fold {f} epoch {ep+1}/{EPOCHS} train loss {tot/len(Xg):.4f}  {time.time()-t0:.0f}s', flush=True)
    net.eval(); outs = []
    with torch.no_grad():
        for i in range(0, len(Xte), 65536):
            outs.append(net(torch.from_numpy(Xte[i:i+65536]).to(dev)).float().cpu().numpy())
    return np.where(te)[0], np.concatenate(outs), (net.module if isinstance(net, nn.DataParallel) else net).state_dict(), (mu, sd)

In [ ]:
oof = b.loc[b['fold'] >= 0, ['city','station','time','anom','fold']].copy()
for name in ['osm_js', 'osm_js+raster']:
    feats = SETS[name]; col = {k: np.full(len(b), np.nan, np.float32) for k in ('mean','q10','q90')}; states = {}
    print('==', name, len(feats), 'features')
    for f in range(5):
        idx, o, sd_, norm = run_fold(feats, f)
        col['mean'][idx], col['q10'][idx], col['q90'][idx] = o[:,0], o[:,1], o[:,2]
        states[f] = {'state': {k: v.cpu() for k, v in sd_.items()}, 'mu': norm[0], 'sd': norm[1]}
    sel = b['fold'].to_numpy() >= 0
    oof[f'mlp_{name}'] = col['mean'][sel]; oof[f'mlp_{name}_q10'] = col['q10'][sel]; oof[f'mlp_{name}_q90'] = col['q90'][sel]
    torch.save({'features': feats, 'folds': states}, f'/content/heatlab_mlp_{name.replace("+","_")}.pt')
oof.to_parquet('/content/mlp_oof.parquet', index=False)
print('saved', oof.shape)

In [ ]:
y = oof['anom'].to_numpy(); res = {'device': dev, 'gpus': int(torch.cuda.device_count()),
     'gpu_name': torch.cuda.get_device_name(0) if dev == 'cuda' else None, 'epochs': EPOCHS, 'batch': BS, 'sets': {}}
for name in ['osm_js', 'osm_js+raster']:
    e = oof[f'mlp_{name}'].to_numpy() - y
    lo, hi = oof[f'mlp_{name}_q10'].to_numpy(), oof[f'mlp_{name}_q90'].to_numpy()
    res['sets'][name] = {'rmse': float(np.sqrt(np.mean(e**2))), 'mae': float(np.abs(e).mean()), 'bias': float(e.mean()),
        'coverage80': float(((y >= lo) & (y <= hi)).mean()), 'width80': float((hi-lo).mean()),
        'by_city': {c: float(np.sqrt(np.mean(e[(oof['city']==c).to_numpy()]**2))) for c in sorted(oof['city'].unique())}}
    print(name, json.dumps(res['sets'][name]))
json.dump(res, open('/content/mlp_metrics.json', 'w'), indent=1)

In [ ]:
from google.colab import files
for fn in ['mlp_oof.parquet', 'mlp_metrics.json', 'heatlab_mlp_osm_js.pt', 'heatlab_mlp_osm_js_raster.pt']: files.download('/content/' + fn)